## Arricchimento dati

Questo script Python esegue la pulizia, la classificazione e l'arricchimento temporale del dataset normativo UE. 

### 1. Caricamento e Standardizzazione

* **Integrazione Dati**: Carica i file relativi a nodi (leggi), archi (citazioni), concetti EuroVoc e le relazioni tra leggi e temi.
* **Pulizia CELEX**: Implementa una funzione di pulizia dei codici **CELEX ID**, rimuovendo caratteri non standard e spazi per garantire l'univocità degli identificatori.

### 2. Estrazione Temporale Avanzata

Per risolvere il problema dei dati mancanti, utilizziamo una logica di fallback a più livelli per determinare l'anno di ogni atto:

* **Analisi Pattern CELEX**: Estrae l'anno direttamente dalla struttura del codice (es. il settore 1 per i Trattati o il settore 3 per i Regolamenti).
* **Fallback URL**: Se l'anno non è presente nel CELEX, lo script lo ricerca all'interno dell'URL del documento.
* **Risultato**: Aumenta drasticamente la copertura temporale, permettendo un'analisi storica completa.

### 3. Classificazione Giuridica e Filtraggio

Categorizziamo ogni documento in base al suo "settore" legale:

* **Identificazione Tipi**: Distingue tra **Regolamenti, Direttive, Decisioni, Trattati** e atti preparatori.
* **Selezione Rilevante**: Filtra il dataset mantenendo solo gli atti normativi primari e secondari, escludendo giurisprudenza o atti nazionali che potrebbero "sporcare" l'analisi della frammentazione legislativa.

### 4. Segmentazione per Ere Storiche

Per facilitare l'analisi dell'evoluzione del diritto UE, i documenti vengono raggruppati in **9 Ere Politiche**, tra cui:

* *Pre-Maastricht*
* *Early/Late 2000s*
* *2020s*
Questo permette di osservare come la struttura del network sia cambiata in risposta ai grandi trattati o alle crisi sistemiche.

### 5. Arricchimento del Network

* **Edge Enrichment**: Non si limita a filtrare i nodi, ma aggiorna anche il file degli archi, inserendo l'anno di origine e di destinazione per ogni citazione. Questo è fondamentale per studiare la direzione del flusso normativo nel tempo.
* **Integrazione EuroVoc**: Mantiene e filtra le connessioni con i concetti tematici EuroVoc.

In [28]:
import pandas as pd
import networkx as nx
import re
import os
import numpy as np
from datetime import datetime

# Configurazione percorsi
raw_path = os.path.join('..', 'data', 'raw')
proc_path = os.path.join('..', 'data', 'processed')
os.makedirs(proc_path, exist_ok=True)

# Carica i dati
print("Caricamento dati...")
nodes = pd.read_csv(os.path.join(raw_path, 'nodes.csv'))
edges = pd.read_csv(os.path.join(raw_path, 'edges.csv'))
eurovoc = pd.read_csv(os.path.join(raw_path, 'eurovoc_concept.csv'))
has_concept = pd.read_csv(os.path.join(raw_path, 'has_concept_edges.csv'))

print(f"Nodes: {len(nodes)} righe")
print(f"Edges: {len(edges)} righe")
print(f"Eurovoc concepts: {len(eurovoc)} righe")
print(f"Has_concept edges: {len(has_concept)} righe")

# 1. PULIZIA E STANDARDIZZAZIONE DEI CELEX ID
def clean_celex(celex):
    """Pulisce e standardizza i CELEX ID"""
    if pd.isna(celex) or celex == '':
        return None
    # Rimuovi spazi e converti in stringa
    celex = str(celex).strip()
    # Alcuni CELEX potrebbero avere formato con slash o trattini
    celex = re.sub(r'[^0-9A-Za-z]', '', celex)
    return celex if celex else None

nodes['celex_clean'] = nodes['celex_id'].apply(clean_celex)

# 2. ESTRAZIONE ANNO ROBUSTA (VERSIONE CORRETTA)
def extract_year_advanced(celex, row=None):
    """
    Estrae l'anno dal CELEX con fallback multipli.
    Versione avanzata che considera diversi formati e fonti alternative.
    Ora con validazione anni plausibili per TUTTI i pattern.
    """
    if pd.isna(celex) or celex == '':
        return None
    
    celex = str(celex)
    current_year = datetime.now().year
    
    # Pattern 1: Trattati (1YYYY...)
    match = re.search(r'^1(\d{4})', celex)
    if match:
        year = int(match.group(1))
        # Anni plausibili per trattati UE (1951-oggi)
        if 1951 <= year <= current_year:
            return year
        else:
            # Se l'anno non è plausibile, potrebbe essere un falso positivo (es. 11195...)
            return None
    
    # Pattern 2: Atti moderni (settore 3-9 + YYYY)
    match = re.search(r'^[3-9](\d{4})', celex)
    if match:
        year = int(match.group(1))
        # Anni plausibili per atti UE (1951-oggi)
        if 1951 <= year <= current_year:
            return year
        else:
            return None
    
    # Pattern 3: CELEX con anno in altre posizioni (es. 32019R0452 -> 2019)
    match = re.search(r'(\d{4})', celex)
    if match:
        year = int(match.group(1))
        # Anni plausibili per diritto UE (1951-oggi)
        if 1951 <= year <= current_year:
            return year
    
    # Pattern 4: Se abbiamo altre info dalla riga (es. URL)
    if row is not None and pd.notna(row.get('url', None)):
        # Cerca anno nell'URL (spesso in formato /YYYY/)
        url_match = re.search(r'/(\d{4})/', str(row['url']))
        if url_match:
            year = int(url_match.group(1))
            if 1951 <= year <= current_year:
                return year
    
    return None

# Applica l'estrazione avanzata
print("\nEstrazione anni...")
nodes['year_extracted'] = nodes.apply(
    lambda row: extract_year_advanced(row['celex_clean'], row), 
    axis=1
)

# 3. ANALISI DELLA COPERTURA TEMPORALE
print("\nAnalisi copertura temporale:")
print(f"Anni originali presenti: {nodes['year'].notna().sum()} su {len(nodes)} ({nodes['year'].notna().sum()/len(nodes)*100:.1f}%)")
print(f"Anni estratti aggiunti: {nodes['year_extracted'].notna().sum()} su {len(nodes)} ({nodes['year_extracted'].notna().sum()/len(nodes)*100:.1f}%)")

# 4. COMBINA LE FONTI
nodes['year_final'] = nodes['year'].fillna(nodes['year_extracted'])
print(f"Anni finali totali: {nodes['year_final'].notna().sum()} su {len(nodes)} ({nodes['year_final'].notna().sum()/len(nodes)*100:.1f}%)")

# 5. CLASSIFICAZIONE DEI TIPI DI ATTO
def classify_legal_type(celex, resource_type):
    """
    Classifica il tipo di atto giuridico in base al CELEX e ai metadati.
    Utile per filtrare solo atti normativi rilevanti.
    """
    if pd.notna(resource_type) and resource_type != 'work' and resource_type != 'Other':
        return resource_type
    
    if pd.isna(celex):
        return 'Unknown'
    
    celex = str(celex)
    
    # Settore 1: Trattati
    if celex.startswith('1'):
        return 'Treaty'
    # Settore 3: Atti legislativi
    elif celex.startswith('3'):
        if 'R' in celex:
            return 'Regulation'
        elif 'L' in celex:
            return 'Directive'
        elif 'D' in celex:
            return 'Decision'
        else:
            return 'Legislative_Act'
    # Settore 4: Atti complementari
    elif celex.startswith('4'):
        return 'Complementary_Act'
    # Settore 5: Atti preparatori
    elif celex.startswith('5'):
        return 'Preparatory_Act'
    # Settore 6: Giurisprudenza
    elif celex.startswith('6'):
        return 'Case_Law'
    # Settore 7: Atti nazionali
    elif celex.startswith('7'):
        return 'National_Act'
    # Settore 8: Relazioni
    elif celex.startswith('8'):
        return 'Report'
    # Settore 9: Altro
    elif celex.startswith('9'):
        return 'Other_Act'
    else:
        return 'Unknown'

print("\nClassificazione tipi di atto...")
nodes['legal_type_class'] = nodes.apply(
    lambda row: classify_legal_type(row['celex_clean'], row['resource_legal_type']), 
    axis=1
)

# 6. STATISTICHE DESCRITTIVE BASE
print("\n=== STATISTICHE DESCRITTIVE ===")
print("\nDistribuzione per tipo di atto (prime 20):")
print(nodes['legal_type_class'].value_counts().head(20))

print("\nDistribuzione temporale (per decade):")
nodes['decade'] = (nodes['year_final'] // 10 * 10).astype('Int64')
print(nodes['decade'].value_counts().sort_index())

# 7. FILTRO PER DOCUMENTI RILEVANTI
# Teniamo solo atti normativi veri e propri
relevant_types = ['Regulation', 'Directive', 'Decision', 'Treaty', 'Legislative_Act', 'Case_Law']
nodes_relevant = nodes[nodes['legal_type_class'].isin(relevant_types)].copy()

print(f"\nDocumenti rilevanti per l'analisi: {len(nodes_relevant)} su {len(nodes)} ({len(nodes_relevant)/len(nodes)*100:.1f}%)")

# 8. CREAZIONE DI UNA COLONNA 'era' PER ANALISI TEMPORALE
def assign_era(year):
    """Assegna un'era storica in base all'anno"""
    if pd.isna(year):
        return 'Unknown'
    if year < 1958:
        return 'Pre-Treaty of Rome'
    elif year < 1987:
        return 'Pre-Single European Act'
    elif year < 1993:
        return 'Pre-Maastricht'
    elif year < 2000:
        return '1990s'
    elif year < 2005:
        return 'Early 2000s'
    elif year < 2010:
        return 'Late 2000s'
    elif year < 2015:
        return 'Early 2010s'
    elif year < 2020:
        return 'Late 2010s'
    else:
        return '2020s'

nodes_relevant['era'] = nodes_relevant['year_final'].apply(assign_era)

# 9. ESPORTAZIONE DATI ARRICCHITI
print("\nEsportazione dati arricchiti...")

# Salva nodes arricchiti
nodes_relevant.to_csv(os.path.join(proc_path, 'nodes_enriched.csv'), index=False)

# Filtra edges per mantenere solo quelli tra nodi rilevanti
relevant_node_ids = set(nodes_relevant[':ID'].unique())
edges_filtered = edges[edges[':START_ID'].isin(relevant_node_ids) & 
                       edges[':END_ID'].isin(relevant_node_ids)]

# Aggiungi informazioni sull'anno per gli edge (utile per analisi temporali)
edges_filtered = edges_filtered.merge(
    nodes_relevant[[':ID', 'year_final']].rename(columns={':ID': ':START_ID', 'year_final': 'start_year'}),
    on=':START_ID',
    how='left'
).merge(
    nodes_relevant[[':ID', 'year_final']].rename(columns={':ID': ':END_ID', 'year_final': 'end_year'}),
    on=':END_ID',
    how='left'
)

edges_filtered.to_csv(os.path.join(proc_path, 'edges_enriched.csv'), index=False)

# Mantieni anche gli HAS_CONCEPT rilevanti
has_concept_filtered = has_concept[has_concept[':START_ID'].isin(relevant_node_ids)]
has_concept_filtered.to_csv(os.path.join(proc_path, 'has_concept_enriched.csv'), index=False)

# 10. STATISTICHE FINALI
print("\n=== STATISTICHE FINALI ===")
print(f"Nodes rilevanti: {len(nodes_relevant)}")
print(f"Edges tra nodi rilevanti: {len(edges_filtered)}")
print(f"Has_concept edges rilevanti: {len(has_concept_filtered)}")

print(f"\nDistribuzione per tipo di atto (rilevanti):")
print(nodes_relevant['legal_type_class'].value_counts())

print(f"\nDistribuzione per era:")
print(nodes_relevant['era'].value_counts().sort_index())

print(f"\nCopertura temporale (rilevanti):")
if nodes_relevant['year_final'].notna().any():
    print(f"Anni coperti: {nodes_relevant['year_final'].min():.0f} - {nodes_relevant['year_final'].max():.0f}")
    print(f"Numero di anni con documenti: {nodes_relevant['year_final'].nunique()}")

# 11. VERIFICA QUALITÀ DATI
print(f"\n=== VERIFICA QUALITÀ DATI ===")
print(f"Nodes con CELEX valido: {nodes_relevant['celex_clean'].notna().sum()}/{len(nodes_relevant)} ({nodes_relevant['celex_clean'].notna().sum()/len(nodes_relevant)*100:.1f}%)")
print(f"Nodes con anno: {nodes_relevant['year_final'].notna().sum()}/{len(nodes_relevant)} ({nodes_relevant['year_final'].notna().sum()/len(nodes_relevant)*100:.1f}%)")
print(f"Nodes con titolo: {nodes_relevant['work_title'].notna().sum()}/{len(nodes_relevant)} ({nodes_relevant['work_title'].notna().sum()/len(nodes_relevant)*100:.1f}%)")

print(f"\nArricchimento completato! Dati salvati in: {proc_path}")

Caricamento dati...
Nodes: 88130 righe
Edges: 219302 righe
Eurovoc concepts: 7613 righe
Has_concept edges: 299026 righe

Estrazione anni...

Analisi copertura temporale:
Anni originali presenti: 34384 su 88130 (39.0%)
Anni estratti aggiunti: 84892 su 88130 (96.3%)
Anni finali totali: 84897 su 88130 (96.3%)

Classificazione tipi di atto...

=== STATISTICHE DESCRITTIVE ===

Distribuzione per tipo di atto (prime 20):
legal_type_class
Regulation           18744
Preparatory_Act      12278
R                    11965
Decision             11717
D                     9740
M                     6628
Unknown               3487
Treaty                2754
Legislative_Act       2659
Directive             2525
Case_Law              2205
L                     1608
Y                      450
Complementary_Act      320
H                      293
A                      201
Q                      107
O                       92
E                       69
B                       69
Name: count, dtype: int64

## Normalizzazione e Ottimizzazione del Dataset (`nodes_light`)

Fase finale di raffinamento dei dati. Il suo obiettivo è duplice: garantire la coerenza tassonomica tra le diverse fonti di dati e produrre una versione "light" del dataset, ottimizzata per le prestazioni computazionali di **Gephi** e degli algoritmi di network analysis.

### 1. Normalizzazione Rigorosa delle Tipologie Legali

Il dataset originale presenta spesso ambiguità nelle etichette (es. "R" vs "Regulation"). Lo script risolve queste incoerenze tramite una funzione di mappatura intelligente:

* **Unificazione Tassonomica**: Converte i codici brevi CELEX e le descrizioni estese in un'unica categoria standardizzata (**Regulation, Directive, Decision, Treaty**).
* **Logica di Fallback**: Se il metadato del tipo di atto è mancante, lo script analizza la struttura del codice CELEX per dedurre la natura giuridica dell'atto.
* **Verifica di Qualità**: Genera una tabella di contingenza (crosstab) per confrontare le classi originali con quelle normalizzate, assicurando che nessun atto rilevante sia andato perduto o classificato erroneamente.

### 2. Selezione delle Variabili Essenziali (Versione "Light")

Per migliorare l'efficienza, lo script isola solo le colonne critiche per l'analisi della **frammentazione tematica**:

* **Identificatori**: ID univoci e CELEX puliti.
* **Metadati Temporali**: Anno finale, decade ed era storica.
* **Dimensioni Tematiche**: Concetti EuroVoc, Domini e Sottodomini (fondamentali per definire i cluster nel network).
* **Referenze**: Titolo dell'opera e URL ufficiale.

### 3. Analisi della Complessità Tematica (EuroVoc)

Lo script introduce una metrica quantitativa per misurare la "multidisciplinarietà" degli atti:

* **Conteggio dei Concetti**: Estrae il numero di concetti EuroVoc associati a ogni legge.
* **Identificazione degli Hub**: Genera una classifica degli atti con il maggior numero di concetti (es. Regolamenti omnibus).
* **Valenza Scientifica**: Un alto numero di concetti EuroVoc spesso correla con un'alta **Betweenness Centrality**, identificando potenziali leggi "ponte" che collegano domini giuridici distanti.

### 4. Statistiche Descrittive Finali

Viene prodotto un report sintetico che analizza:

* **Distribuzione Temporale**: Frequenza dei tipi di atto per decade, utile per contestualizzare la produzione normativa (es. l'esplosione dei Regolamenti negli ultimi anni).
* **Integrità del Dataset**: Verifica la copertura percentuale di titoli, anni e domini EuroVoc.

### Output

Il risultato è il file `nodes_light.csv`, un dataset pulito e leggero che garantisce:

1. **Caricamento rapido** in Gephi (riduzione del rumore visivo e della RAM occupata).
2. **Etichette coerenti** per la colorazione dei nodi nel grafo (Partition).
3. **Dati pronti** per la creazione di legende professionali nella tesi.

In [16]:
import pandas as pd
import os

# Percorsi
proc_path = os.path.join('..', 'data', 'processed')

# Carica i dati arricchiti
nodes_relevant = pd.read_csv(os.path.join(proc_path, 'nodes_enriched.csv'))

print(f"Colonne disponibili: {list(nodes_relevant.columns)}")
print(f"Shape: {nodes_relevant.shape}")

# 1. NORMALIZZAZIONE DEI TIPI DI ATTO
print("\n=== NORMALIZZAZIONE TIPI DI ATTO ===")

# Mappa dei codici CELEX ai tipi normalizzati
celex_type_map = {
    'R': 'Regulation',
    'L': 'Directive', 
    'D': 'Decision',
    # Aggiungi altri se necessario
}

def normalize_legal_type(row):
    """
    Normalizza i tipi di atto unificando codici CELEX e tipi estesi
    """
    tipo_originale = row['legal_type_class']
    celex = row['celex_clean'] if pd.notna(row['celex_clean']) else ''
    
    # 1. Se è già un tipo esteso, tienilo
    if tipo_originale in ['Regulation', 'Directive', 'Decision', 'Treaty', 'Legislative_Act']:
        return tipo_originale
    
    # 2. Se è un codice CELEX nella mappa
    if tipo_originale in celex_type_map:
        return celex_type_map[tipo_originale]
    
    # 3. Altrimenti, cerca nel CELEX
    if pd.notna(celex):
        celex_str = str(celex)
        if 'R' in celex_str and tipo_originale not in ['Regulation', 'R']:
            return 'Regulation'
        elif 'L' in celex_str and tipo_originale not in ['Directive', 'L']:
            return 'Directive'
        elif 'D' in celex_str and tipo_originale not in ['Decision', 'D']:
            return 'Decision'
    
    # 4. Se non si riesce a normalizzare, tieni l'originale
    return tipo_originale

# Applica la normalizzazione
nodes_relevant['legal_type_normalized'] = nodes_relevant.apply(normalize_legal_type, axis=1)

# Mostra il confronto
print("\nConfronto prima/dopo normalizzazione:")
comparison = pd.crosstab(
    nodes_relevant['legal_type_class'], 
    nodes_relevant['legal_type_normalized'],
    margins=True
)
print(comparison)

# 2. CREAZIONE VERSIONE LEGGERA (con i nomi esatti delle colonne)
print("\n=== CREAZIONE VERSIONE LEGGERA ===")

essential_cols = [
    ':ID',
    'celex_id',
    'celex_clean',
    'work_title',
    'year_final',
    'legal_type_normalized',  
    'era',
    'decade',
    'eurovoc_concepts:STRING[]',  
    'domains:STRING[]',          
    'subdomains:STRING[]',    
    'url'
]

optional_cols = [
    'sector',
    'sector_label',
    'resource_legal_type',
    'legal_type_class'  
]

# Verifica quali colonne esistono effettivamente
available_cols = []
for col in essential_cols + optional_cols:
    if col in nodes_relevant.columns:
        available_cols.append(col)
    else:
        print(f"Attenzione: colonna '{col}' non trovata")

print(f"\nColonne selezionate: {available_cols}")

# Crea versione leggera
nodes_light = nodes_relevant[available_cols].copy()

# 3. STATISTICHE SUI TITOLI
print("\n=== ANALISI TITOLI ===")
print(f"Titoli presenti: {nodes_light['work_title'].notna().sum()}/{len(nodes_light)}")
print(f"Titoli vuoti: {nodes_light['work_title'].isna().sum()}/{len(nodes_light)}")

# Mostra un esempio di riga con titolo se esiste
if nodes_light['work_title'].notna().any():
    sample = nodes_light[nodes_light['work_title'].notna()].iloc[0]
    print(f"\nEsempio titolo: {sample['work_title']}")
    print(f"CELEX: {sample['celex_id']}")

# 4. STATISTICHE SUI TIPI NORMALIZZATI
print("\n=== DISTRIBUZIONE TIPI NORMALIZZATI ===")
print(nodes_light['legal_type_normalized'].value_counts())

print("\n=== DISTRIBUZIONE PER DECADE (tipi normalizzati) ===")
print(pd.crosstab(
    nodes_light['decade'], 
    nodes_light['legal_type_normalized'],
    margins=True
))

# 5. ANALISI CONCETTI EUROVOC (se presenti)
print("\n=== ANALISI CONCETTI EUROVOC ===")

# Funzione per contare concetti
def count_concepts(concept_string):
    if pd.isna(concept_string):
        return 0
    return len(str(concept_string).split(';'))

nodes_light['n_eurovoc_concepts'] = nodes_light['eurovoc_concepts:STRING[]'].apply(count_concepts)

print(f"Media concetti Eurovoc per atto: {nodes_light['n_eurovoc_concepts'].mean():.1f}")
print(f"Mediana concetti Eurovoc: {nodes_light['n_eurovoc_concepts'].median():.0f}")
print(f"Max concetti Eurovoc: {nodes_light['n_eurovoc_concepts'].max()}")

# Mostra gli atti con più concetti
top_concepts = nodes_light.nlargest(5, 'n_eurovoc_concepts')[
    ['celex_id', 'work_title', 'n_eurovoc_concepts', 'legal_type_normalized', 'year_final']
]
print("\nAtti con più concetti Eurovoc:")
print(top_concepts.to_string(index=False))

# 6. ESPORTAZIONE
print("\n=== ESPORTAZIONE ===")
nodes_light.to_csv(os.path.join(proc_path, 'nodes_light.csv'), index=False)
print(f"Versione leggera salvata: {len(nodes_light)} righe, {len(nodes_light.columns)} colonne")

# 7. VERIFICA FINALE QUALITÀ
print("\n=== VERIFICA FINALE QUALITÀ DATI ===")
print(f"Totale nodi: {len(nodes_light)}")
print(f"Con anno: {nodes_light['year_final'].notna().sum()} ({nodes_light['year_final'].notna().sum()/len(nodes_light)*100:.1f}%)")
print(f"Con CELEX: {nodes_light['celex_clean'].notna().sum()} ({nodes_light['celex_clean'].notna().sum()/len(nodes_light)*100:.1f}%)")
print(f"Con concetti Eurovoc: {nodes_light['eurovoc_concepts:STRING[]'].notna().sum()} ({nodes_light['eurovoc_concepts:STRING[]'].notna().sum()/len(nodes_light)*100:.1f}%)")
print(f"Con domini: {nodes_light['domains:STRING[]'].notna().sum()} ({nodes_light['domains:STRING[]'].notna().sum()/len(nodes_light)*100:.1f}%)")

Colonne disponibili: [':ID', 'cellar_id', ':LABEL', 'planjo_id', 'oj_id', 'immc_id', 'celex_id', 'eli_id', 'consil_id', 'ecb_id', 'uriserv_id', 'work_title', 'sector', 'year', 'resource_legal_type', 'list_types:STRING[]', 'list_work_has_resource_type:STRING[]', 'pages_total:FLOAT', 'eurovoc_concepts:STRING[]', 'domains:STRING[]', 'subdomains:STRING[]', 'url', 'sectorHierarchy', 'sector_label', 'celex_clean', 'year_extracted', 'year_final', 'legal_type_class', 'decade', 'era']
Shape: (38399, 30)

=== NORMALIZZAZIONE TIPI DI ATTO ===

Confronto prima/dopo normalizzazione:
legal_type_normalized  Decision  Directive  Legislative_Act  Regulation  \
legal_type_class                                                          
Decision                  11717          0                0           0   
Directive                     0       2525                0           0   
Legislative_Act               0          0             2659           0   
Regulation                    0          0      

## Identificazione semantica dei nodi seed

Identificazione di un sotto-insieme di atti normativi definiti **"Seed" (semi)**, focalizzati sul tema del **Golden Power** e degli investimenti diretti esteri (FDI). 

### 1. Livello 1: Integrazione della Conoscenza Esperta

Il processo inizia con l'inserimento manuale di "atti fondamentali" universalmente riconosciuti come pilastri della materia:

* **Base Trattati**: Articoli chiave del TFUE sulla libera circolazione dei capitali e il diritto di stabilimento (es. Art. 63 e 65 TFUE).
* **Legislazione Chiave**: Il Regolamento (UE) 2019/452 sul controllo degli investimenti diretti esteri.
* **Giurisprudenza Storica**: Inserimento delle sentenze della Corte di Giustizia riguardanti le "Golden Share" (es. C-58/99 contro l'Italia), essenziali per comprendere l'evoluzione giuridica del limite al controllo statale.

### 2. Livello 2: Espansione Semantica via EuroVoc

Per superare i limiti di una ricerca puramente manuale, lo script utilizza un'**espansione basata su parole chiave** attraverso il tesauro EuroVoc:

* **Mappatura Settoriale**: Definisce dizionari di termini per i settori critici soggetti a Golden Power: Difesa, Energia, Trasporti, Comunicazioni, Cybersecurity, Agrifood e Semiconduttori.
* **Filtro per Domini Prioritari**: I concetti trovati vengono filtrati attraverso domini istituzionali specifici (es. *Finance, Trade, Energy, Industry*) per eliminare rumore semantico e mantenere solo le accezioni giuridicamente rilevanti.

### 3. Livello 3: Estrazione degli Atti Correlati

Attraverso la relazione `HAS_CONCEPT`, lo script identifica automaticamente tutti gli atti nel database che sono indicizzati con i concetti EuroVoc selezionati al passo precedente.

* **Analisi Temporale e Tipologica**: Viene generata una distribuzione per decade e per tipo di atto dei nuovi nodi identificati, verificando la coerenza storica del campione (es. l'aumento degli atti legati alla sicurezza cibernetica negli anni 2020).

### 4. Creazione del Grafo Focale (Contesto di 1° Grado)

L'analisi non si limita ai soli nodi "seed", ma si espande per includere il loro contesto relazionale:

* **Espansione ai Vicini**: Vengono estratti tutti i nodi che citano o sono citati dai "seed" (**1st-degree neighbors**).
* **Definizione del Grafo Focale**: L'unione dei nodi seed e dei loro vicini costituisce il "Grafo Focale", l'area specifica del network normativo UE dove si analizza la frammentazione tematica relativa alla protezione degli asset strategici.

### Risultati e Statistiche Finali

Lo script salva tre file fondamentali in `data/processed`:

1. `seed_concepts.csv`: L'elenco dei concetti EuroVoc che definiscono l'area del Golden Power.
2. `seed_works.csv`: La lista degli atti normativi "core" della ricerca.
3. `focal_nodes.csv`: Il dataset completo (seed + vicini) pronto per l'importazione in **Gephi** per l'analisi della **Modularity** e della **Centrality**.

In [24]:
import pandas as pd
import os
import re

# Percorsi
proc_path = os.path.join('..', 'data', 'processed')
raw_path = os.path.join('..', 'data', 'raw')

# Carica i dati
print("=== PASSO 2: IDENTIFICAZIONE SEMI GOLDEN POWER ===\n")

nodes = pd.read_csv(os.path.join(proc_path, 'nodes_light.csv'))
has_concept = pd.read_csv(os.path.join(proc_path, 'has_concept_enriched.csv'))
eurovoc = pd.read_csv(os.path.join(raw_path, 'eurovoc_concept.csv'))
edges = pd.read_csv(os.path.join(proc_path, 'edges_enriched.csv'))

print(f"Nodes leggeri: {len(nodes)}")
print(f"Has_concept edges: {len(has_concept)}")
print(f"Eurovoc concepts: {len(eurovoc)}")
print(f"Edges: {len(edges)}")

# 1. ATTI FONDAMENTALI NOTI (manuali)
print("\n=== Livello 1: Atti fondamentali noti ===")

# Trattati rilevanti (libertà di circolazione, stabilimento)
treaty_seeds = [
    '11957E052',  # Art. 52 TCE - diritto di stabilimento
    '11957E056',  # Art. 56 TCE - libera circolazione capitali
    '11957E073D', # Art. 73D TCE - eccezioni per ordine pubblico
    '11992E073B', # Art. 73B Maastricht - movimenti capitali
    '11992E073D', # Art. 73D Maastricht - eccezioni
    '12016E063',  # Art. 63 TFUE - libera circolazione capitali
    '12016E065',  # Art. 65 TFUE - eccezioni
]

# Regolamento FDI screening
fdi_regulation = ['32019R0452']  # Reg. (UE) 2019/452

# Atti su golden share (giurisprudenza chiave)
golden_share_cases = [
    '61999CJ0058',  # Causa C-58/99 (Commissione c. Italia)
    '62007CJ0326',  # Causa C-326/07 (Commissione c. Italia)
    '62000CJ0463',  # Causa C-463/00 (Commissione c. Spagna)
    '62002CJ0174',  # Causa C-174/02 (Commissione c. Paesi Bassi)
    '62006CJ0274',  # Causa C-274/06 (Commissione c. Spagna)
]

# Unisci tutti i semi noti
known_seeds = treaty_seeds + fdi_regulation + golden_share_cases

# Trova questi nodi nel dataset
seed_nodes_known = nodes[nodes['celex_clean'].isin(known_seeds)].copy()
print(f"Trovati {len(seed_nodes_known)} atti fondamentali su {len(known_seeds)} cercati")

if len(seed_nodes_known) > 0:
    print("\nAtti fondamentali trovati:")
    for _, row in seed_nodes_known.iterrows():
        concepts = row['eurovoc_concepts:STRING[]'] if pd.notna(row['eurovoc_concepts:STRING[]']) else 'N/A'
        print(f"  - {row['celex_clean']} ({row['legal_type_normalized']}, {row['year_final']:.0f})")
        if concepts != 'N/A':
            print(f"    Concetti: {concepts[:100]}...")

# 2. CONCETTI EUROVOC RILEVANTI
print("\n=== Livello 2: Concetti Eurovoc per settore ===")

# Mappa dei settori Golden Power con parole chiave (dalla ricerca)
golden_sectors = {
    # Difesa e sicurezza (art. 1 D.L. 21/2012)
    'defence': ['defence', 'military', 'security', 'defense'],
    'national_security': ['national security', 'state security', 'public safety'],
    'public_order': ['public order', 'public policy', 'ordre public'],

    # Settori tradizionali (art. 2 D.L. 21/2012)
    'energy': ['energy', 'electricity', 'gas', 'oil', 'power grid'],
    'transport': ['transport', 'infrastructure', 'airport', 'port', 'railway'],
    'communications': ['communication', 'telecom', 'telecommunication', 'broadband', '5G'],

    # Settori espansi (DPCM 179/2020, Reg. 452/2019)
    'financial': ['financial', 'banking', 'credit', 'insurance', 'investment services'],
    'health': ['health', 'healthcare', 'sanitary', 'medical', 'pharmaceutical'],
    'agrifood': ['agriculture', 'food', 'agrifood', 'farming', 'supply chain'],
    'semiconductors': ['semiconductor', 'microchip', 'electronics', 'integrated circuit'],
    'cybersecurity': ['cybersecurity', 'cyber security', 'information security', 'data protection'],
    'critical_infrastructure': ['critical infrastructure', 'strategic asset', 'essential facility'],
    'technology': ['technology', 'innovation', 'research', 'development', 'high-tech'],

    # Concetti giuridici chiave (dalla giurisprudenza)
    'capital_movement': ['capital movement', 'free movement of capital', 'capital flow'],
    'establishment': ['right of establishment', 'freedom of establishment'],
    'foreign_investment': ['foreign investment', 'foreign direct investment', 'FDI', 'inward investment'],
    'golden_share': ['golden share', 'special right', 'special power', 'state control'],
}

# Prepara tutte le parole chiave
all_keywords = []
for sector, keywords in golden_sectors.items():
    all_keywords.extend(keywords)

# Cerca nei nomi dei concetti (usando case=False invece di (?i))
pattern = '|'.join([re.escape(k) for k in all_keywords if len(k) > 3])  # escapa caratteri speciali
print(f"\nCerca pattern: {pattern[:100]}...")

seed_concepts = eurovoc[eurovoc['name'].str.contains(pattern, na=False, regex=True, case=False)]

# FIX: Cerca per codici Eurovoc numerici SOLO se esistono keyword numeriche
# (Il bug precedente: join su lista vuota produce '', che matcha tutto con str.contains)
digit_keywords = [k for k in all_keywords if k.isdigit()]
if digit_keywords:
    digit_pattern = '|'.join([f'/{k}' for k in digit_keywords])
    seed_concepts_codes = eurovoc[eurovoc['id:ID'].str.contains(digit_pattern, na=False, regex=False)]
else:
    # Nessuna keyword numerica: restituisci DataFrame vuoto con stessa struttura
    seed_concepts_codes = eurovoc.iloc[0:0].copy()
    print("  (Nessuna keyword numerica trovata - seed_concepts_codes vuoto)")

# Unisci e rimuovi duplicati
seed_concepts_all = pd.concat([seed_concepts, seed_concepts_codes]).drop_duplicates(subset=['id:ID'])

# DIAGNOSTICA (prima del filtro domini)
print(f"\n=== DIAGNOSTICA ===")
print(f"seed_concepts (da pattern): {len(seed_concepts)}")
print(f"seed_concepts_codes (da codici): {len(seed_concepts_codes)}")
print(f"seed_concepts_all (unione): {len(seed_concepts_all)}")
print("\nPrimi 20 concetti (prima del filtro domini):")
for _, row in seed_concepts_all.head(20).iterrows():
    print(f"  - {row['name']} | Domini: {row['domains:STRING[]']}")
all_domains_before = []
for domains in seed_concepts_all['domains:STRING[]'].dropna():
    all_domains_before.extend([d.strip() for d in str(domains).split(';') if d.strip()])
print("\nTop 10 domini prima del filtro:")
print(pd.Series(all_domains_before).value_counts().head(10))

# --- FILTRO PER DOMINI PRIORITARI ---
priority_domains = [
    '04 POLITICS',
    '08 INTERNATIONAL RELATIONS',
    '10 EUROPEAN UNION',
    '12 LAW',
    '20 TRADE',
    '24 FINANCE',
    '28 SOCIAL QUESTIONS',
    '32 EDUCATION AND COMMUNICATIONS',
    '48 TRANSPORT',
    '56 AGRICULTURE, FORESTRY AND FISHERIES',
    '60 AGRI-FOODSTUFFS',
    '64 PRODUCTION, TECHNOLOGY AND RESEARCH',
    '66 ENERGY',
    '68 INDUSTRY'
]

def in_priority_domains(domains_str):
    if pd.isna(domains_str):
        return False
    domains = [d.strip() for d in str(domains_str).split(';') if d.strip()]
    return any(d in priority_domains for d in domains)

# Applica il filtro e sovrascrivi seed_concepts_all con la versione filtrata
seed_concepts_all['in_priority'] = seed_concepts_all['domains:STRING[]'].apply(in_priority_domains)
seed_concepts_filtered = seed_concepts_all[seed_concepts_all['in_priority']].copy()
seed_concepts_all = seed_concepts_filtered.drop(columns=['in_priority'])

# VERIFICA FILTRO (dopo il filtro domini)
print(f"\n=== VERIFICA FILTRO ===")
print(f"seed_concepts_all (dopo filtro): {len(seed_concepts_all)}")
print(f"Di cui con domini nulli: {seed_concepts_all['domains:STRING[]'].isna().sum()}")
all_domains_after = []
for domains in seed_concepts_all['domains:STRING[]'].dropna():
    all_domains_after.extend([d.strip() for d in str(domains).split(';') if d.strip()])
print("\nTop 10 domini dopo il filtro:")
print(pd.Series(all_domains_after).value_counts().head(10))

print(f"\nTrovati {len(seed_concepts_all)} concetti Eurovoc rilevanti (dopo filtro domini)")

# Analizza i domini dei concetti trovati
print("\nTop 20 concetti trovati (per nome):")
for _, row in seed_concepts_all.head(20).iterrows():
    print(f"  - {row['name']} ({row['id:ID']})")

# Estrai domini
def extract_domains(domains_str):
    if pd.isna(domains_str):
        return []
    return [d.strip() for d in str(domains_str).split(';') if d.strip()]

all_domains = []
for domains in seed_concepts_all['domains:STRING[]'].dropna():
    all_domains.extend(extract_domains(domains))

domain_counts = pd.Series(all_domains).value_counts()
print("\nDistribuzione per dominio:")
for domain, count in domain_counts.head(10).items():
    print(f"  {domain}: {count} concetti")

# Salva i concetti seed
seed_concepts_all.to_csv(os.path.join(proc_path, 'seed_concepts.csv'), index=False)

# 3. TROVA ATTI CHE HANNO QUESTI CONCETTI
print("\n=== Livello 3: Atti collegati ai concetti seed ===")

# Filtra has_concept per mantenere solo i concetti seed
seed_concept_ids = set(seed_concepts_all['id:ID'])
has_concept_seed = has_concept[has_concept[':END_ID'].isin(seed_concept_ids)]

# Trova gli atti collegati
seed_work_ids = set(has_concept_seed[':START_ID'])
seed_works_by_concept = nodes[nodes[':ID'].isin(seed_work_ids)].copy()

print(f"Trovati {len(seed_works_by_concept)} atti collegati ai concetti seed")

# Analisi temporale
print("\nDistribuzione temporale atti seed (per decade):")
seed_by_decade = seed_works_by_concept['decade'].value_counts().sort_index()
for decade, count in seed_by_decade.items():
    decade_str = f"{decade:.0f}" if pd.notna(decade) else "Unknown"
    print(f"  {decade_str}: {count} atti ({count/len(seed_works_by_concept)*100:.1f}%)")

# Distribuzione per tipo
print("\nTipi di atto nei seed:")
type_dist = seed_works_by_concept['legal_type_normalized'].value_counts()
for typ, count in type_dist.items():
    print(f"  {typ}: {count} ({count/len(seed_works_by_concept)*100:.1f}%)")

# 4. UNISCI TUTTI I SEED
print("\n=== Unisci tutti i seed ===")

# Combina atti noti e atti da concetti
all_seed_works = pd.concat([
    seed_nodes_known,
    seed_works_by_concept
]).drop_duplicates(subset=[':ID'])

print(f"Totale atti seed unici: {len(all_seed_works)}")

# Funzione per ottenere i concetti di un atto
def get_work_concepts(work_id):
    concepts = has_concept_seed[has_concept_seed[':START_ID'] == work_id][':END_ID'].tolist()
    if not concepts:
        return ''
    concept_names = eurovoc[eurovoc['id:ID'].isin(concepts)]['name'].tolist()
    return ';'.join(concept_names[:5])  # primi 5 per leggibilità

all_seed_works['seed_concepts'] = all_seed_works[':ID'].apply(get_work_concepts)

# Aggiungi conteggio concetti
all_seed_works['n_seed_concepts'] = all_seed_works[':ID'].apply(
    lambda x: len(has_concept_seed[has_concept_seed[':START_ID'] == x])
)

# Salva
all_seed_works.to_csv(os.path.join(proc_path, 'seed_works.csv'), index=False)

print(f"\nSeed salvati in: {os.path.join(proc_path, 'seed_works.csv')}")

# Mostra statistiche
print(f"\nStatistiche seed:")
print(f"  Media concetti per atto: {all_seed_works['n_seed_concepts'].mean():.1f}")
print(f"  Mediana concetti: {all_seed_works['n_seed_concepts'].median():.0f}")
print(f"  Max concetti: {all_seed_works['n_seed_concepts'].max()}")

print("\nEsempi di atti seed (con più concetti):")
top_seeds = all_seed_works.nlargest(10, 'n_seed_concepts')[
    ['celex_clean', 'year_final', 'legal_type_normalized', 'n_seed_concepts', 'seed_concepts']
]
for _, row in top_seeds.iterrows():
    concepts = row['seed_concepts'][:80] + ('...' if len(row['seed_concepts']) > 80 else '')
    print(f"  {row['celex_clean']} ({row['year_final']:.0f}) - {row['legal_type_normalized']} - {row['n_seed_concepts']} concetti")
    if concepts:
        print(f"    {concepts}")

# 5. ESPANDI AI VICINI (opzionale - per contesto)
print("\n=== Espansione ai vicini (1 salto) ===")

# Trova tutti gli atti citati DAI seed
cited_by_seed = edges[edges[':START_ID'].isin(all_seed_works[':ID'])]
cited_nodes = set(cited_by_seed[':END_ID'])

# Trova tutti gli atti che citano I seed
citing_seed = edges[edges[':END_ID'].isin(all_seed_works[':ID'])]
citing_nodes = set(citing_seed[':START_ID'])

# Unisci
neighbor_ids = cited_nodes.union(citing_nodes)
neighbors = nodes[nodes[':ID'].isin(neighbor_ids)].copy()

print(f"Vicini di 1° grado:")
print(f"  Citati dai seed: {len(cited_nodes)}")
print(f"  Citano i seed: {len(citing_nodes)}")
print(f"  Totale vicini unici: {len(neighbors)}")

# Crea il grafo focale (seed + vicini)
focal_nodes = pd.concat([
    all_seed_works,
    neighbors
]).drop_duplicates(subset=[':ID'])

print(f"\nGrafo focale totale: {len(focal_nodes)} nodi")

# Salva
focal_nodes.to_csv(os.path.join(proc_path, 'focal_nodes.csv'), index=False)

# 6. STATISTICHE RIASSUNTIVE
print("\n=== RIEPILOGO FINALE ===")
print(f"Concetti Eurovoc seed: {len(seed_concepts_all)}")
print(f"Atti seed (conoscenza diretta): {len(seed_nodes_known)}")
print(f"Atti seed (da concetti): {len(seed_works_by_concept)}")
print(f"Atti seed (totali unici): {len(all_seed_works)}")
print(f"Vicini di 1° grado: {len(neighbors)}")
print(f"Grafo focale totale: {len(focal_nodes)}")

print("\nDistribuzione per tipo nel grafo focale:")
focal_type_dist = focal_nodes['legal_type_normalized'].value_counts()
for typ, count in focal_type_dist.items():
    print(f"  {typ}: {count} ({count/len(focal_nodes)*100:.1f}%)")

print("\nDistribuzione per decade nel grafo focale:")
focal_decade_dist = focal_nodes['decade'].value_counts().sort_index()
for decade, count in focal_decade_dist.items():
    if pd.notna(decade):
        print(f"  {decade:.0f}: {count} ({count/len(focal_nodes)*100:.1f}%)")

=== PASSO 2: IDENTIFICAZIONE SEMI GOLDEN POWER ===

Nodes leggeri: 38399
Has_concept edges: 137980
Eurovoc concepts: 7613
Edges: 64142

=== Livello 1: Atti fondamentali noti ===
Trovati 7 atti fondamentali su 13 cercati

Atti fondamentali trovati:
  - 32019R0452 (Regulation, 2019)
    Concetti: powers of the institutions (EU);04 POLITICS;08 INTERNATIONAL RELATIONS;implementation of the budget;...
  - 12016E063 (Treaty, 2016)
  - 12016E065 (Treaty, 2016)
  - 11957E056 (Treaty, 1957)
  - 11957E052 (Treaty, 1957)
  - 11992E073B (Treaty, 1992)
  - 11992E073D (Treaty, 1992)

=== Livello 2: Concetti Eurovoc per settore ===

Cerca pattern: defence|military|security|defense|national\ security|state\ security|public\ safety|public\ order|pu...
  (Nessuna keyword numerica trovata - seed_concepts_codes vuoto)

=== DIAGNOSTICA ===
seed_concepts (da pattern): 738
seed_concepts_codes (da codici): 0
seed_concepts_all (unione): 738

Primi 20 concetti (prima del filtro domini):
  - 32 EDUCATION AND COM

## Estrazione della Network Focalizzato (Golden Power)

Strategia di filtraggio semantico "ristretto" per isolare i nodi e le relazioni che costituiscono il cuore tematico della tesi. L'obiettivo è ridurre la complessità del grafo globale di 4.427 nodi a un **Grafo Focale** coerente e analizzabile.

### 1. Definizione dei "Semi" Normativi (Seed Nodes)

Il processo di identificazione avviene su tre livelli di profondità:

* **Livello Fondamentale**: Inclusione manuale di atti "pivot", tra cui i Trattati sulle libertà di circolazione (Art. 63 TFUE), il Regolamento FDI Screening (2019/452) e le sentenze storiche della Corte di Giustizia sulle *Golden Share*.
* **Livello Semantico Ristretto**: Utilizzo di un pattern di ricerca specifico (es. *cybersecurity, critical infrastructure, strategic asset*) per identificare i concetti EuroVoc che definiscono il perimetro del Golden Power moderno.
* **Filtro per Domini Prioritari**: I concetti identificati vengono filtrati per domini istituzionali rilevanti (es. *Finance, Trade, Energy, Transport*), eliminando i risultati non pertinenti alla sicurezza e agli investimenti.

### 2. Espansione Relazionale e Vicinato

Per non perdere il contesto normativo, lo script espande il network oltre i soli atti "seed":

* **Analisi dei Vicini (1-hop)**: Vengono estratti tutti i nodi che citano o sono citati dai semi normativi. Questo permette di catturare la "legislazione di contorno" che supporta o limita i poteri speciali dello Stato.
* **Grafo Focale**: L'unione dei semi e dei loro vicini costituisce il dataset finale per l'analisi della frammentazione, garantendo che i cluster individuati non siano isolati ma connessi alla struttura legale complessiva.

### 3. Preparazione per l'Analisi in Gephi

Lo script automatizza la generazione dei file pronti per l'importazione nel software di network analysis:

* **gephi_nodes_focal.csv**: Include metadati armonizzati come l'Era storica, l'Anno e il Tipo di atto (Regulation, Directive, ecc.).
* **gephi_edges_focal.csv**: Contiene tutte le relazioni (citazioni) interne al sotto-network selezionato.
* **Attributi Analitici**: Viene aggiunta una colonna `Seed_Concepts` che indica quanti concetti chiave del Golden Power sono legati a ogni legge, parametro utile per scalare la dimensione dei nodi durante la visualizzazione.

### Risultati della Selezione

L'output finale fornisce statistiche immediate sulla composizione del grafo focalizzato:

* **Distribuzione per Tipo**: Analisi della prevalenza di Regolamenti o Direttive nel settore della sicurezza.
* **Evoluzione per Decade**: Monitoraggio dell'intensità normativa nel tempo, utile per supportare l'ipotesi di una crescente frammentazione tematica nell'era digitale (2020s).

In [29]:
import pandas as pd
import os
import re

# Percorsi
proc_path = os.path.join('..', 'data', 'processed')
raw_path = os.path.join('..', 'data', 'raw')

# Carica i dati
print("=== PASSO 2 (RISTRETTO): IDENTIFICAZIONE SEMI GOLDEN POWER ===\n")

nodes = pd.read_csv(os.path.join(proc_path, 'nodes_light.csv'))
has_concept = pd.read_csv(os.path.join(proc_path, 'has_concept_enriched.csv'))
eurovoc = pd.read_csv(os.path.join(raw_path, 'eurovoc_concept.csv'))
edges = pd.read_csv(os.path.join(proc_path, 'edges_enriched.csv'))

# 1. ATTI FONDAMENTALI NOTI (uguali)
print("\n=== Livello 1: Atti fondamentali noti ===")
treaty_seeds = [
    '11957E052', '11957E056', '11957E073D',
    '11992E073B', '11992E073D',
    '12016E063', '12016E065',
]
fdi_regulation = ['32019R0452']
golden_share_cases = [
    '61999CJ0058', '62007CJ0326', '62000CJ0463',
    '62002CJ0174', '62006CJ0274'
]
known_seeds = treaty_seeds + fdi_regulation + golden_share_cases
seed_nodes_known = nodes[nodes['celex_clean'].isin(known_seeds)].copy()
print(f"Trovati {len(seed_nodes_known)} atti fondamentali")

# 2. CONCETTI EUROVOC MIRATI
print("\n=== Livello 2: Concetti Eurovoc mirati ===")

# Pattern RISTRETTO - solo termini specifici della Golden Power
golden_pattern = '|'.join([
    # Settori chiave (specifici)
    'defence', 'military', 'armament',
    'energy policy', 'energy supply',
    'transport policy', 'infrastructure',
    'telecommunications', '5G', 'broadband',
    'financial institution', 'banking', 'insurance',
    'public health', 'healthcare',
    'agri-food', 'food supply',
    'semiconductor', 'microchip',
    'cybersecurity', 'information security',
    # Concetti giuridici specifici
    'foreign investment', 'direct investment',
    'golden share', 'special right', 'state control',
    'national security', 'public order', 'ordre public',
    'critical infrastructure', 'strategic asset',
    'free movement of capital', 'capital movement',
    'right of establishment', 'freedom of establishment'
])

print(f"Pattern ristretto: {golden_pattern[:100]}...")

# Cerca nei nomi - e fa una COPIA immediata!
seed_concepts = eurovoc[eurovoc['name'].str.contains(
    golden_pattern, na=False, regex=True, case=False
)].copy()  # <--- AGGIUNTO .copy() QUI

print(f"\nTrovati {len(seed_concepts)} concetti Eurovoc specifici")

print("\nTop concetti trovati:")
for _, row in seed_concepts.head(20).iterrows():
    print(f"  - {row['name']}")

# 3. DOMINI PRIORITARI (filtra per rilevanza)
print("\n=== Filtro per domini prioritari ===")

# Domini realmente rilevanti per Golden Power
priority_domains = [
    '04 POLITICS',           # ordine pubblico, sicurezza
    '08 INTERNATIONAL RELATIONS',  # relazioni estere, FDI
    '10 EUROPEAN UNION',     # mercato interno, libertà
    '12 LAW',                # diritto societario, proprietà
    '20 TRADE',              # commercio, investimenti
    '24 FINANCE',            # banche, assicurazioni
    '28 SOCIAL QUESTIONS',   # salute, protezione civile
    '32 EDUCATION AND COMMUNICATIONS',  # telecomunicazioni, 5G
    '48 TRANSPORT',          # infrastrutture
    '56 AGRICULTURE, FORESTRY AND FISHERIES',# agroalimentare
    '60 AGRI-FOODSTUFFS',    # filiera alimentare
    '64 PRODUCTION, TECHNOLOGY AND RESEARCH',  # semiconduttori, AI
    '66 ENERGY',             # energia, reti
    '68 INDUSTRY'            # industria strategica
]

# Funzione per controllare se un concetto è in domini prioritari
def in_priority_domains(domains_str):
    if pd.isna(domains_str):
        return False
    domains = [d.strip() for d in str(domains_str).split(';')]
    return any(d in priority_domains for d in domains)

# Ora possiamo modificare seed_concepts perché è una copia indipendente
seed_concepts['in_priority'] = seed_concepts['domains:STRING[]'].apply(in_priority_domains)
seed_concepts_filtered = seed_concepts[seed_concepts['in_priority']].copy()

print(f"Concetti in domini prioritari: {len(seed_concepts_filtered)}")

# Mostra distribuzione per dominio
print("\nDistribuzione per dominio (filtrati):")
domain_counts = {}
for domains in seed_concepts_filtered['domains:STRING[]'].dropna():
    for d in str(domains).split(';'):
        d = d.strip()
        if d in priority_domains:
            domain_counts[d] = domain_counts.get(d, 0) + 1

for domain, count in sorted(domain_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {domain}: {count} concetti")

# 4. ATTI COLLEGATI AI CONCETTI FILTRATI
print("\n=== Livello 3: Atti collegati ai concetti filtrati ===")

seed_concept_ids = set(seed_concepts_filtered['id:ID'])
has_concept_seed = has_concept[has_concept[':END_ID'].isin(seed_concept_ids)]
seed_work_ids = set(has_concept_seed[':START_ID'])
seed_works_by_concept = nodes[nodes[':ID'].isin(seed_work_ids)].copy()

print(f"Trovati {len(seed_works_by_concept)} atti collegati")

# 5. UNISCI CON ATTI NOTI
all_seed_works = pd.concat([
    seed_nodes_known,
    seed_works_by_concept
]).drop_duplicates(subset=[':ID'])

print(f"\nTotale atti seed unici: {len(all_seed_works)}")

# 6. ESPANDI AI VICINI (opzionale)
print("\n=== Espansione ai vicini (1 salto) ===")
cited_by_seed = edges[edges[':START_ID'].isin(all_seed_works[':ID'])]
cited_nodes = set(cited_by_seed[':END_ID'])
citing_seed = edges[edges[':END_ID'].isin(all_seed_works[':ID'])]
citing_nodes = set(citing_seed[':START_ID'])
neighbor_ids = cited_nodes.union(citing_nodes)
neighbors = nodes[nodes[':ID'].isin(neighbor_ids)].copy()

focal_nodes = pd.concat([
    all_seed_works,
    neighbors
]).drop_duplicates(subset=[':ID'])

print(f"Grafo focale totale: {len(focal_nodes)} nodi")

# 7. STATISTICHE FINALI
print("\n=== STATISTICHE GRAFO FOCALE ===")
print(f"Distribuzione per tipo:")
print(focal_nodes['legal_type_normalized'].value_counts())

print(f"\nDistribuzione per decade:")
print(focal_nodes['decade'].value_counts().sort_index())

print(f"\nPercentuale del totale originale: {len(focal_nodes)/len(nodes)*100:.1f}%")

# 8. PREPARAZIONE PER GEPHI
print("\n=== Preparazione file per Gephi ===")

# Nodi per Gephi
gephi_nodes = focal_nodes[[
    ':ID', 'celex_clean', 'year_final', 'legal_type_normalized', 'era'
]].copy()
gephi_nodes.rename(columns={
    ':ID': 'Id',
    'celex_clean': 'Label',
    'year_final': 'Year',
    'legal_type_normalized': 'Type',
    'era': 'Era'
}, inplace=True)

# Aggiungi conteggio concetti seed
gephi_nodes['Seed_Concepts'] = gephi_nodes['Id'].apply(
    lambda x: len(has_concept_seed[has_concept_seed[':START_ID'] == x])
)

# Edges per Gephi
gephi_edges = edges[
    edges[':START_ID'].isin(focal_nodes[':ID']) & 
    edges[':END_ID'].isin(focal_nodes[':ID'])
].copy()
gephi_edges.rename(columns={
    ':START_ID': 'Source',
    ':END_ID': 'Target',
    ':TYPE': 'Type'
}, inplace=True)

# Salva
gephi_nodes.to_csv(os.path.join(proc_path, 'gephi_nodes_focal.csv'), index=False)
gephi_edges.to_csv(os.path.join(proc_path, 'gephi_edges_focal.csv'), index=False)

print(f"\nFile salvati in {proc_path}:")
print(f"  - gephi_nodes_focal.csv ({len(gephi_nodes)} nodi)")
print(f"  - gephi_edges_focal.csv ({len(gephi_edges)} archi)")

=== PASSO 2 (RISTRETTO): IDENTIFICAZIONE SEMI GOLDEN POWER ===


=== Livello 1: Atti fondamentali noti ===
Trovati 7 atti fondamentali

=== Livello 2: Concetti Eurovoc mirati ===
Pattern ristretto: defence|military|armament|energy policy|energy supply|transport policy|infrastructure|telecommunicat...

Trovati 126 concetti Eurovoc specifici

Top concetti trovati:
  - 60 AGRI-FOODSTUFFS
  - 0816 international security
  - 0821 defence
  - 2416 financial institutions and credit
  - 2421 free movement of capital
  - 2431 insurance
  - 4806 transport policy
  - 6031 agri-foodstuffs
  - 6606 energy policy
  - insurance indemnity
  - telecommunications industry
  - transport infrastructure
  - industrial infrastructure
  - financial institution
  - direct investment
  - foreign investment
  - military court
  - self-defence
  - free movement of capital
  - insurance contract

=== Filtro per domini prioritari ===
Concetti in domini prioritari: 123

Distribuzione per dominio (filtrati):
  24 FI